## Step 1: Import Libraries & Load Data

In [1]:
# Step 1: Import Libraries & Load Data
import pandas as pd
import numpy as np
from scipy import stats
import json
import time
import logging
import random
from datetime import datetime
from groq import Groq
import os
import faiss
from sentence_transformers import SentenceTransformer

# Load data
patients = pd.read_csv('healthcare_patients.csv')
encounters = pd.read_csv('healthcare_encounters.csv')

# Keep latest encounter per patient
latest_encounters = encounters.sort_values('date').groupby('patient_id').last().reset_index()

# Join
df = patients.merge(latest_encounters, on='patient_id', how='left')

# Fill missing encounters
df['dx_code'] = df['dx_code'].fillna('NONE')
df['note'] = df['note'].fillna('No clinical encounter recorded.')

print(f"Patients: {len(patients)}")
print(f"Encounters: {len(encounters)}")
print(f"Merged records: {len(df)}")
print(f"Missing values after merge:\n{df.isnull().sum()}")
print(f"\nSample:\n{df.head(3)}")

/Users/saravananshanmugam/capstone-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Patients: 400
Encounters: 1500
Merged records: 400
Missing values after merge:
patient_id      0
age             0
sex             0
bmi             0
smoker          0
diabetic        0
encounter_id    5
date            5
dx_code         0
note            0
dtype: int64

Sample:
  patient_id  age sex   bmi  smoker  diabetic encounter_id        date  \
0     H10000   68   M  25.3       0         0       V00485  2024-03-15   
1     H10001   70   M  23.4       1         0       V00255  2024-04-16   
2     H10002   41   M  24.5       0         0       V00959  2024-05-01   

  dx_code                                               note  
0     E78  Patient presents with headache for 1 days. His...  
1     J45  Reports headache episodically; denies chest pa...  
2     M54  Follow-up visit. hyperlipidemia controlled. Re...  


In [2]:
print(patients.head())
print(encounters.head())
print(f"Patients: {patients.shape[0]}, Encounters: {encounters.shape[0]}")
print(f"Patients columns: {patients.columns.tolist()}")
print(f"Encounters columns: {encounters.columns.tolist()}")
print(f"Patients data types:\n{patients.dtypes}")
print(f"Encounters data types:\n{encounters.dtypes}")
print(f"Patients unique values:\n{patients.nunique()}")
print(f"Encounters unique values:\n{encounters.nunique()}")
print(f"Patients description:\n{patients.describe(include='all')}")
print(f"Encounters description:\n{encounters.describe(include='all')}")
# print shape of dataframes
print(f"Patients shape: {patients.shape}")
print(f"Encounters shape: {encounters.shape}")

  patient_id  age sex   bmi  smoker  diabetic
0     H10000   68   M  25.3       0         0
1     H10001   70   M  23.4       1         0
2     H10002   41   M  24.5       0         0
3     H10003   44   M  16.2       0         1
4     H10004   77   F  33.3       0         0
  encounter_id patient_id        date dx_code  \
0       V00676     H10279  2024-01-01     I10   
1       V00715     H10068  2024-01-01     J45   
2       V00951     H10046  2024-01-01     M54   
3       V00240     H10235  2024-01-01     I25   
4       V00695     H10096  2024-01-01     E78   

                                                note  
0  Follow-up visit. sciatica controlled. Recommen...  
1  Reports back pain episodically; denies chest p...  
2  Follow-up visit. GERD controlled. Recommending...  
3  Patient presents with fatigue for 7 days. Hist...  
4  Reports cough episodically; denies chest pain....  
Patients: 400, Encounters: 1500
Patients columns: ['patient_id', 'age', 'sex', 'bmi', 'smoker', 'di

In [3]:
# check for missing values
print("Missing values in patients dataset:")
print(patients.isnull().sum())
print("\nMissing values in encounters dataset:")
print(encounters.isnull().sum())    
# Basic statistics
print("\nBasic statistics for patients dataset:")
print(patients.describe())
print("\nBasic statistics for encounters dataset:")
print(encounters.describe())

Missing values in patients dataset:
patient_id    0
age           0
sex           0
bmi           0
smoker        0
diabetic      0
dtype: int64

Missing values in encounters dataset:
encounter_id    0
patient_id      0
date            0
dx_code         0
note            0
dtype: int64

Basic statistics for patients dataset:
              age        bmi      smoker    diabetic
count  400.000000  400.00000  400.000000  400.000000
mean    54.212500   26.79775    0.192500    0.157500
std     19.783726    4.66833    0.394757    0.364728
min     21.000000   13.10000    0.000000    0.000000
25%     38.000000   23.50000    0.000000    0.000000
50%     53.500000   26.30000    0.000000    0.000000
75%     71.000000   29.62500    0.000000    0.000000
max     89.000000   40.00000    1.000000    1.000000

Basic statistics for encounters dataset:
       encounter_id patient_id        date dx_code  \
count          1500       1500        1500    1500   
unique         1500        395         200    

In [4]:
# Keep latest encounter per patient
latest_encounters = encounters.sort_values('date').groupby('patient_id').last().reset_index()

# Join
df = patients.merge(latest_encounters, on='patient_id', how='left')

# ── Data Overview ──
print("=" * 50)
print("PATIENTS DATASET")
print("=" * 50)
print(f"Shape: {patients.shape}")
print(f"Columns: {list(patients.columns)}")
print(f"Missing values:\n{patients.isnull().sum()}")
print(f"\nData Types:\n{patients.dtypes}")
print(f"\nBasic Stats:\n{patients.describe()}")

print("\n" + "=" * 50)
print("ENCOUNTERS DATASET")
print("=" * 50)
print(f"Shape: {encounters.shape}")
print(f"Columns: {list(encounters.columns)}")
print(f"Missing values:\n{encounters.isnull().sum()}")
print(f"\nUnique patients in encounters: {encounters['patient_id'].nunique()}")
print(f"Date range: {encounters['date'].min()} to {encounters['date'].max()}")
print(f"Top 5 diagnosis codes:\n{encounters['dx_code'].value_counts().head()}")

print("\n" + "=" * 50)
print("MERGED DATASET")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing values after merge:\n{df.isnull().sum()}")
print(f"\nSample records:")
print(df.head(3))

PATIENTS DATASET
Shape: (400, 6)
Columns: ['patient_id', 'age', 'sex', 'bmi', 'smoker', 'diabetic']
Missing values:
patient_id    0
age           0
sex           0
bmi           0
smoker        0
diabetic      0
dtype: int64

Data Types:
patient_id        str
age             int64
sex               str
bmi           float64
smoker          int64
diabetic        int64
dtype: object

Basic Stats:
              age        bmi      smoker    diabetic
count  400.000000  400.00000  400.000000  400.000000
mean    54.212500   26.79775    0.192500    0.157500
std     19.783726    4.66833    0.394757    0.364728
min     21.000000   13.10000    0.000000    0.000000
25%     38.000000   23.50000    0.000000    0.000000
50%     53.500000   26.30000    0.000000    0.000000
75%     71.000000   29.62500    0.000000    0.000000
max     89.000000   40.00000    1.000000    1.000000

ENCOUNTERS DATASET
Shape: (1500, 5)
Columns: ['encounter_id', 'patient_id', 'date', 'dx_code', 'note']
Missing values:
encou

In [5]:
# Step 1b: PII Validation & Anonymization Confirmation

# Data Analysis — Full PII Check Across All 400 Records

import re

print("=" * 50)
print(f"FULL PII SCAN — {len(df)} RECORDS")
print("=" * 50)

# Check 1 — patient_id format
all_valid_ids = df['patient_id'].str.match(r'^H\d{5}$').all()
print(f"\nPatient ID format (H#####)     : {'ALL VALID' if all_valid_ids else 'INVALID IDs FOUND'}")

# Check 2 — email addresses
email_pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
emails_found = df['note'].apply(lambda x: bool(email_pattern.search(str(x)))).sum()
print(f"Email addresses in notes       : {emails_found} found")

# Check 3 — phone numbers
phone_pattern = re.compile(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b')
phones_found = df['note'].apply(lambda x: bool(phone_pattern.search(str(x)))).sum()
print(f"Phone numbers in notes         : {phones_found} found")

# Check 4 — name patterns
name_pattern = re.compile(r'\b(Mr\.|Mrs\.|Dr\.|Ms\.)\s[A-Z][a-z]+')
names_found = df['note'].apply(lambda x: bool(name_pattern.search(str(x)))).sum()
print(f"Name patterns in notes         : {names_found} found")

# Check 5 — unique note patterns to confirm templated
unique_starts = df['note'].apply(lambda x: str(x).split('.')[0]).nunique()
print(f"\nUnique note opening sentences  : {unique_starts}")
print(f"Total notes                    : {len(df)}")
print(f"Templated ratio                : {unique_starts/len(df)*100:.1f}% unique openings")

print("\n" + "=" * 50)
if emails_found == 0 and phones_found == 0 and names_found == 0 and all_valid_ids:
    print("RESULT: NO PII DETECTED ACROSS ALL 400 RECORDS")
else:
    print("WARNING: POTENTIAL PII DETECTED — REVIEW REQUIRED")

FULL PII SCAN — 400 RECORDS

Patient ID format (H#####)     : ALL VALID
Email addresses in notes       : 0 found
Phone numbers in notes         : 0 found
Name patterns in notes         : 0 found

Unique note opening sentences  : 96
Total notes                    : 400
Templated ratio                : 24.0% unique openings

RESULT: NO PII DETECTED ACROSS ALL 400 RECORDS


In [6]:

print("=" * 50)
print("PII VALIDATION — DATA SAFETY CHECK")
print("=" * 50)

# Confirm dataset is synthetic and anonymized
print("\nDataset Source  : Provided as course material — confirmed anonymized via full dataset scan")
print("Patient IDs       : Anonymized format (H10000 to H10399)")
print("Clinical Notes    : No PII detected — template-generated, no real names or identifiers found")
print("External API Use  : Anonymized patient profile and clinical notes sent to Groq API for LLM inference only")
print("Data Governance   : No data stored or used for training by Groq API provider")

# Confirm ID format
all_valid = df['patient_id'].str.match(r'^H\d{5}$').all()
print(f"\nPatient ID format validation : {'PASSED' if all_valid else 'FAILED'}")
print(f"Total records validated      : {len(df)}")

print("\nAnonymization Status:")
print(f"  Patient IDs    : Pre-anonymized (sequential H##### codes)")
print(f"  Clinical Notes : No PII detected — template-generated, no real names or identifiers found")
print(f"  Sex            : Encoded as M/F only — no full names")
print(f"  Age            : Numeric only — no date of birth")
print(f"  BMI            : Numeric only — no biometric identifiers")
print(f"\nCONCLUSION: Dataset is pre-anonymized and requires no further masking.")

print("\nProduction Note:")
print("In a real-world deployment, runtime PII scrubbing would be implemented")
print("using tools such as Microsoft Presidio or AWS Comprehend Medical")
print("before any data is transmitted to an external API.")

print("\nPII CHECK COMPLETE — Dataset safe for use in this project.")

PII VALIDATION — DATA SAFETY CHECK

Dataset Source  : Provided as course material — confirmed anonymized via full dataset scan
Patient IDs       : Anonymized format (H10000 to H10399)
Clinical Notes    : No PII detected — template-generated, no real names or identifiers found
External API Use  : Anonymized patient profile and clinical notes sent to Groq API for LLM inference only
Data Governance   : No data stored or used for training by Groq API provider

Patient ID format validation : PASSED
Total records validated      : 400

Anonymization Status:
  Patient IDs    : Pre-anonymized (sequential H##### codes)
  Clinical Notes : No PII detected — template-generated, no real names or identifiers found
  Sex            : Encoded as M/F only — no full names
  Age            : Numeric only — no date of birth
  BMI            : Numeric only — no biometric identifiers

CONCLUSION: Dataset is pre-anonymized and requires no further masking.

Production Note:
In a real-world deployment, runtime 

Data looks clean. Key observations before we move to Step 2:

400 patients, 1500 encounters, merged to 400 rows — correct
5 patients have no encounter record — we'll handle those with a default note
Top dx codes: F41 (anxiety), I25 (coronary artery disease), E78 (cholesterol), G47 (sleep disorders), I10 (hypertension) — good mix for risk scoring
BMI ranges 13.1 to 40.0 — the 13.1 will likely trigger our anomaly detection
19.2% smokers, 15.7% diabetic in the cohort

## Step 2: Build Patient Text for Embedding

In [7]:
# Step 2: Build Patient Text for Embedding

def build_patient_text(row):
    smoker_text = "smoker" if row['smoker'] == 1 else "non-smoker"
    diabetic_text = "diabetic" if row['diabetic'] == 1 else "not diabetic"
    return (
        f"Patient {row['patient_id']}, age {row['age']}, {row['sex']}, "
        f"BMI {row['bmi']}, {smoker_text}, {diabetic_text}. "
        f"Diagnosis code: {row['dx_code']}. "
        f"Clinical note: {row['note']}"
    )

df['patient_text'] = df.apply(build_patient_text, axis=1)

print(f"Total records ready for embedding: {len(df)}")
print(f"Missing in patient_text: {df['patient_text'].isnull().sum()}")
print(f"\nSample patient text:\n{df['patient_text'][0]}")

Total records ready for embedding: 400
Missing in patient_text: 0

Sample patient text:
Patient H10000, age 68, M, BMI 25.3, non-smoker, not diabetic. Diagnosis code: E78. Clinical note: Patient presents with headache for 1 days. History of diabetes. Vitals stable.


## Step 3: Generate Embeddings Using Sentence Transformer

In [8]:
# Step 3: Generate Embeddings Using Sentence Transformer

print("Loading embedding model: all-MiniLM-L6-v2")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for all 400 patients...")
embeddings = embed_model.encode(
    df['patient_text'].tolist(),
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype('float32')

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Each patient represented as a {embeddings.shape[1]}-dimensional vector")
print(f"Sample embedding (first 5 dimensions of patient 1): {embeddings[0][:5]}")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9483.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for all 400 patients...


Batches: 100%|██████████| 13/13 [00:01<00:00, 10.43it/s]


Embedding shape: (400, 384)
Each patient represented as a 384-dimensional vector
Sample embedding (first 5 dimensions of patient 1): [-0.02327132  0.08685286 -0.09027811  0.01571661 -0.08493029]


## Step 4: Build FAISS Index for Similarity Search

In [9]:
# Step 4: Build FAISS Index for Similarity Search

dimension = embeddings.shape[1]  # 384

# Build FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index built successfully")
print(f"Total vectors indexed: {index.ntotal}")
print(f"Vector dimensions: {dimension}")

# Test similarity search — find 3 most similar patients to patient 0
test_embedding = embeddings[0].reshape(1, -1)
distances, indices = index.search(test_embedding, 4)  # 4 because first result is itself

print(f"\nSimilarity search test — Top 3 similar patients to {df['patient_id'][0]}:")
for i, (dist, idx) in enumerate(zip(distances[0][1:], indices[0][1:])):
    print(f"  {i+1}. {df['patient_id'][idx]} | Age: {df['age'][idx]} | BMI: {df['bmi'][idx]} | Distance: {dist:.4f}")

FAISS index built successfully
Total vectors indexed: 400
Vector dimensions: 384

Similarity search test — Top 3 similar patients to H10000:
  1. H10176 | Age: 76 | BMI: 26.9 | Distance: 0.0962
  2. H10071 | Age: 38 | BMI: 26.6 | Distance: 0.1962
  3. H10302 | Age: 62 | BMI: 27.8 | Distance: 0.2011


## Step 5: Connect Groq API & Test Connection

In [ ]:
# Step 5: Connect Groq API & Test Connection

os.environ["GROQ_API_KEY"] = "removed before exporting as html"  # replace with your key
client = Groq()

# Test with single patient
test_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
#    model="llama-3.2-3b-preview", --- model_decommissioned
    messages=[
        {"role": "user", "content": f"Summarize this patient record in one sentence for a business audience: {df['patient_text'][0]}"}
    ],
    max_tokens=100
)

print("Groq connection successful!")
print(f"Model: llama-3.1-8b-instant")
print(f"Test response: {test_response.choices[0].message.content}")

Groq connection successful!
Model: llama-3.1-8b-instant
Test response: Patient H10000, a 68-year-old, non-diabetic male with a stable vitals profile, visited for a one-day headache, and a history of diabetes is noted, despite the patient himself being non-diabetic.


## Step 6: Risk Scoring

In [13]:
# Step 6: Risk Scoring

def compute_risk_score(row):
    score = 0.0

    # Age risk
    if row['age'] >= 70: score += 0.30
    elif row['age'] >= 55: score += 0.20
    elif row['age'] >= 40: score += 0.10

    # BMI risk
    if row['bmi'] >= 35: score += 0.20
    elif row['bmi'] >= 30: score += 0.10
    elif row['bmi'] >= 25: score += 0.05  # overweight
    elif row['bmi'] >= 16: score += 0.00  # normal weight
    elif row['bmi'] < 16: score += 0.15   # severely underweight

    # Smoker
    if row['smoker'] == 1: score += 0.20

    # Diabetic
    if row['diabetic'] == 1: score += 0.20

    # Diagnosis code risk
    high_risk_dx = ['I25', 'I10', 'E11', 'I21', 'I50']
    medium_risk_dx = ['E78', 'G47', 'F41', 'J45', 'M54']
    if row['dx_code'] in high_risk_dx: score += 0.20
    elif row['dx_code'] in medium_risk_dx: score += 0.10

    return round(min(score, 1.0), 2)

def classify_risk(score):
    if score >= 0.70: return 'HIGH'
    elif score >= 0.40: return 'MEDIUM'
    else: return 'LOW'

df['risk_score'] = df.apply(compute_risk_score, axis=1)
df['risk_level'] = df['risk_score'].apply(classify_risk)

total = len(df)
high = (df['risk_level'] == 'HIGH').sum()
medium = (df['risk_level'] == 'MEDIUM').sum()
low = (df['risk_level'] == 'LOW').sum()

print("Risk Score Distribution:")
print(f"HIGH   : {high}  ({high/total*100:.2f}%)")
print(f"MEDIUM : {medium} ({medium/total*100:.2f}%)")
print(f"LOW    : {low} ({low/total*100:.2f}%)")
print(f"\nAvg Risk Score : {df['risk_score'].mean():.2f}")
print(f"Max Risk Score : {df['risk_score'].max():.2f}")
print(f"Min Risk Score : {df['risk_score'].min():.2f}")
print(f"\nSample output:")
print(df[['patient_id', 'age', 'bmi', 'smoker', 'diabetic', 'dx_code', 'risk_score', 'risk_level']].head(10))

Risk Score Distribution:
HIGH   : 27  (6.75%)
MEDIUM : 195 (48.75%)
LOW    : 178 (44.50%)

Avg Risk Score : 0.39
Max Risk Score : 0.90
Min Risk Score : 0.00

Sample output:
  patient_id  age   bmi  smoker  diabetic dx_code  risk_score risk_level
0     H10000   68  25.3       0         0     E78        0.35        LOW
1     H10001   70  23.4       1         0     J45        0.60     MEDIUM
2     H10002   41  24.5       0         0     M54        0.20        LOW
3     H10003   44  16.2       0         1     K21        0.30        LOW
4     H10004   77  33.3       0         0     F41        0.50     MEDIUM
5     H10005   53  28.1       0         0     G47        0.25        LOW
6     H10006   46  26.3       0         0     I10        0.35        LOW
7     H10007   80  23.5       0         0     I10        0.50     MEDIUM
8     H10008   24  28.6       0         0     G47        0.15        LOW
9     H10009   67  31.4       1         1     E11        0.90       HIGH


## Step 7: Anomaly Detection

In [14]:
# Step 7: Anomaly Detection

# Z-score based anomaly detection on BMI and risk score
df['bmi_zscore'] = np.abs(stats.zscore(df['bmi']))
df['risk_zscore'] = np.abs(stats.zscore(df['risk_score']))

# Flag anomalies — exceeds 2.5 standard deviations
df['anomaly'] = (
    (df['bmi_zscore'] > 2.5) |
    (df['risk_zscore'] > 2.5)
)

# Silent risk — no high risk dx_code but risk score >= 0.60
high_risk_dx = ['I25', 'I10', 'E11', 'I21', 'I50']
df['silent_risk'] = (
    (~df['dx_code'].isin(high_risk_dx)) &
    (df['risk_score'] >= 0.60)
)

print(f"Total anomalies flagged : {df['anomaly'].sum()}")
print(f"Silent risk profiles   : {df['silent_risk'].sum()}")

print(f"\nAnomalies:")
print(df[df['anomaly']][['patient_id', 'age', 'bmi', 'bmi_zscore', 
                          'risk_score', 'risk_zscore', 'dx_code', 'risk_level']])

print(f"\nSilent Risk Profiles (sample):")
print(df[df['silent_risk']][['patient_id', 'age', 'bmi', 'smoker', 
                              'diabetic', 'dx_code', 'risk_score', 'risk_level']].head(10))

Total anomalies flagged : 11
Silent risk profiles   : 31

Anomalies:
    patient_id  age   bmi  bmi_zscore  risk_score  risk_zscore dx_code  \
9       H10009   67  31.4    0.987080        0.90     2.837077     E11   
12      H10012   72  28.4    0.343647        0.85     2.560018     J45   
33      H10033   74  24.2    0.557159        0.90     2.837077     E11   
51      H10051   89  31.6    1.029975        0.90     2.837077     G47   
52      H10052   86  13.1    2.937861        0.55     0.897669     M54   
192     H10192   54  39.8    2.788692        0.60     1.174727     G47   
292     H10292   70  35.5    1.866438        0.90     2.837077     E11   
300     H10300   43  38.7    2.552766        0.50     0.620611     I25   
315     H10315   41  14.8    2.573249        0.35     0.210564     E78   
320     H10320   44  39.7    2.767244        0.50     0.620611     I25   
329     H10329   23  40.0    2.831587        0.30     0.487623     G47   

    risk_level  
9         HIGH  
12      

## Privacy & Responsible AI Notes

### Data Privacy
- Patient records use anonymized IDs (e.g., H10000) — no real PII is present in this dataset.
- Clinical notes are synthetic and do not contain identifiable information.
- In a production environment, all patient data would be encrypted at rest and in transit before being sent to any external API.

### Responsible LLM Use
- LLM summaries are generated only for HIGH and MEDIUM risk patients. LOW risk patients are excluded to reduce unnecessary inference and API cost. This is a documented design assumption.
- The Groq API is used for inference only — no patient data is stored or used for model training by the API provider.
- All LLM outputs are treated as decision-support tools, not final clinical decisions. Human review is mandatory before any action is taken.

### LLMOps Guardrails
- Embedding drift is monitored using KL-divergence across runs. Retraining is recommended if drift exceeds 0.25 for two consecutive runs.
- Model confidence is tracked per inference. Low confidence outputs should be flagged for manual review.

## Step 8: RAG Integration & LLM-Generated Summaries

In [16]:
# Step 8: RAG Integration & LLM-Generated Summaries (FAISS-powered)

def get_similar_cases_faiss(patient_idx, k=3):
    query_embedding = embeddings[patient_idx].reshape(1, -1)
    distances, indices = index.search(query_embedding, k + 1)
    similar = []
    for dist, idx in zip(distances[0][1:], indices[0][1:]):
        similar.append({
            'patient_id': df['patient_id'][idx],
            'age': df['age'][idx],
            'risk_score': df['risk_score'][idx],
            'note': df['note'][idx],
            'distance': round(float(dist), 4)
        })
    return similar

def generate_llm_summary(row, similar_cases):
    similar_text = "\n".join([
        f"- Patient {s['patient_id']}, age {s['age']}, risk score {s['risk_score']}: {s['note']}"
        for s in similar_cases
    ])

    prompt = f"""You are a health risk analyst preparing a report for a business audience (underwriters/clinical ops team).

Current Patient:
{row['patient_text']}
Risk Score: {row['risk_score']} ({row['risk_level']})
Anomaly: {row['anomaly']}
Silent Risk: {row['silent_risk']}

Similar Historical Cases retrieved via embedding similarity:
{similar_text}

Write a 3-sentence plain language risk summary. No medical jargon. Focus on what the business team needs to act on."""

    # Measure real latency
    start_time = time.time()

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )

    end_time = time.time()
    real_latency = round(end_time - start_time, 2)

    # Real token count from API response
    real_tokens = response.usage.completion_tokens

    summary_text = response.choices[0].message.content

    return summary_text, real_latency, real_tokens

# Run on all HIGH and MEDIUM risk patients
# flagged_patients = df[df['risk_level'].isin(['HIGH', 'MEDIUM'])].copy()
flagged_patients = df.copy()
all_summaries = []

# print(f"Generating summaries for {len(flagged_patients)} patients...")
print(f"Generating summaries for {len(flagged_patients)} patients (all risk levels)...")

for i, (idx, row) in enumerate(flagged_patients.iterrows()):
    patient_idx = df.index.get_loc(idx)
    similar = get_similar_cases_faiss(patient_idx)
    summary, latency, tokens = generate_llm_summary(row, similar)

    all_summaries.append({
        'patient_id': row['patient_id'],
        'age': row['age'],
        'sex': row['sex'],
        'risk_score': row['risk_score'],
        'risk_level': row['risk_level'],
        'anomaly': row['anomaly'],
        'silent_risk': row['silent_risk'],
        'dx_code': row['dx_code'],
        'summary': summary,
        'latency_sec': latency,
        'tokens_used': tokens
    })
    time.sleep(0.5)

    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/{len(flagged_patients)}...")

summaries_df = pd.DataFrame(all_summaries)
print(f"\nDone. Total summaries generated: {len(summaries_df)}")
print(f"Avg real latency : {summaries_df['latency_sec'].mean():.2f}s")
print(f"Avg real tokens  : {summaries_df['tokens_used'].mean():.0f}")
print(f"\nSample HIGH risk summary:")
sample = summaries_df[summaries_df['risk_level'] == 'HIGH'].iloc[0]
print(f"Patient: {sample['patient_id']} | Score: {sample['risk_score']}")
print(f"Latency: {sample['latency_sec']}s | Tokens: {sample['tokens_used']}")
print(f"Summary: {sample['summary']}")

Generating summaries for 400 patients (all risk levels)...
Processed 20/400...
Processed 40/400...
Processed 60/400...
Processed 80/400...
Processed 100/400...
Processed 120/400...
Processed 140/400...
Processed 160/400...
Processed 180/400...
Processed 200/400...
Processed 220/400...
Processed 240/400...
Processed 260/400...
Processed 280/400...
Processed 300/400...
Processed 320/400...
Processed 340/400...
Processed 360/400...
Processed 380/400...
Processed 400/400...

Done. Total summaries generated: 400
Avg real latency : 2.88s
Avg real tokens  : 91

Sample HIGH risk summary:
Patient: H10009 | Score: 0.9
Latency: 0.52s | Tokens: 119
Summary: Patient H10009 is at high risk for potential health complications and has a high chance of increasing healthcare costs due to his age, BMI, smoking status, and diabetes. Historically, similar patients with similar characteristics have seen an increase in health issues, suggesting that this patient could experience similar health challenges unle

In [ ]:
# # Step 8: RAG Integration & LLM-Generated Summaries (FAISS-powered) - simulated

# def get_similar_cases_faiss(patient_idx, k=3):
#     query_embedding = embeddings[patient_idx].reshape(1, -1)
#     distances, indices = index.search(query_embedding, k + 1)  # +1 because first result is itself
#     similar = []
#     for dist, idx in zip(distances[0][1:], indices[0][1:]):
#         similar.append({
#             'patient_id': df['patient_id'][idx],
#             'age': df['age'][idx],
#             'risk_score': df['risk_score'][idx],
#             'note': df['note'][idx],
#             'distance': round(float(dist), 4)
#         })
#     return similar

# def generate_llm_summary(row, similar_cases):
#     similar_text = "\n".join([
#         f"- Patient {s['patient_id']}, age {s['age']}, risk score {s['risk_score']}: {s['note']}"
#         for s in similar_cases
#     ])

#     prompt = f"""You are a health risk analyst preparing a report for a business audience (underwriters/clinical ops team).

# Current Patient:
# {row['patient_text']}
# Risk Score: {row['risk_score']} ({row['risk_level']})
# Anomaly: {row['anomaly']}
# Silent Risk: {row['silent_risk']}

# Similar Historical Cases retrieved via embedding similarity:
# {similar_text}

# Write a 3-sentence plain language risk summary. No medical jargon. Focus on what the business team needs to act on."""

#     response = client.chat.completions.create(
#         model="llama-3.1-8b-instant",
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=200
#     )
#     return response.choices[0].message.content

# # Run on all HIGH and MEDIUM risk patients
# flagged_patients = df[df['risk_level'].isin(['HIGH', 'MEDIUM'])].copy()
# all_summaries = []

# print(f"Generating summaries for {len(flagged_patients)} patients...")

# for i, (idx, row) in enumerate(flagged_patients.iterrows()):
#     patient_idx = df.index.get_loc(idx)
#     similar = get_similar_cases_faiss(patient_idx)
#     summary = generate_llm_summary(row, similar)
#     all_summaries.append({
#         'patient_id': row['patient_id'],
#         'age': row['age'],
#         'sex': row['sex'],
#         'risk_score': row['risk_score'],
#         'risk_level': row['risk_level'],
#         'anomaly': row['anomaly'],
#         'silent_risk': row['silent_risk'],
#         'dx_code': row['dx_code'],
#         'summary': summary
#     })
#     time.sleep(0.5)

#     if (i + 1) % 20 == 0:
#         print(f"Processed {i+1}/{len(flagged_patients)}...")

# summaries_df = pd.DataFrame(all_summaries)
# print(f"\nDone. Total summaries generated: {len(summaries_df)}")
# print(f"\nSample HIGH risk summary:")
# sample = summaries_df[summaries_df['risk_level'] == 'HIGH'].iloc[0]
# print(f"Patient: {sample['patient_id']} | Score: {sample['risk_score']}")
# print(f"Summary: {sample['summary']}")

## Step 9: LLMOps Logging & Drift Monitoring

In [17]:
# Step 9: LLMOps Logging & Drift Monitoring

log_records = []

for _, row in summaries_df.iterrows():
    log_records.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'patient_id': row['patient_id'],
        'risk_score': row['risk_score'],
        'risk_level': row['risk_level'],
        'anomaly': row['anomaly'],
        'silent_risk': row['silent_risk'],
        'latency_sec': row['latency_sec'],
        'tokens_used': row['tokens_used'],
        'model': 'llama-3.1-8b-instant'
    })

log_df = pd.DataFrame(log_records)

# Drift monitoring — illustrative across 7 runs
# Note: In production KL-divergence would be calculated against a 
# baseline embedding distribution updated weekly. Values below are 
# illustrative of expected drift patterns in a production pipeline.
drift_scores = [0.08, 0.09, 0.13, 0.11, 0.18, 0.14, 0.12]
drift_status = 'STABLE' if max(drift_scores) < 0.25 else 'ALERT'

print("=" * 50)
print("LLMOPS PIPELINE SUMMARY")
print("=" * 50)
print(f"Total inferences run     : {len(log_df)}")
print(f"Model                    : llama-3.1-8b-instant")
print(f"Embedding model          : all-MiniLM-L6-v2 (384 dimensions)")
print(f"Vector store             : FAISS (IndexFlatL2)")
print(f"Avg real latency         : {log_df['latency_sec'].mean():.2f}s")
print(f"Avg real tokens per call : {log_df['tokens_used'].mean():.0f}")
print(f"Confidence scoring       : Not available — logprobs not supported by llama-3.1-8b-instant on Groq")
print(f"Anomalies flagged        : {log_df['anomaly'].sum()}")
print(f"Silent risk profiles     : {log_df['silent_risk'].sum()}")
print(f"Embedding drift (latest) : {drift_scores[-1]} (illustrative)")
print(f"Drift status             : {drift_status} (threshold 0.25)")
print(f"Drift scores across runs : {drift_scores}")
print(f"\nSample Log Entries:")
print(log_df[['patient_id', 'risk_level', 'latency_sec',
              'tokens_used', 'anomaly']].head(10))

LLMOPS PIPELINE SUMMARY
Total inferences run     : 400
Model                    : llama-3.1-8b-instant
Embedding model          : all-MiniLM-L6-v2 (384 dimensions)
Vector store             : FAISS (IndexFlatL2)
Avg real latency         : 2.88s
Avg real tokens per call : 91
Confidence scoring       : Not available — logprobs not supported by llama-3.1-8b-instant on Groq
Anomalies flagged        : 11
Silent risk profiles     : 31
Embedding drift (latest) : 0.12 (illustrative)
Drift status             : STABLE (threshold 0.25)
Drift scores across runs : [0.08, 0.09, 0.13, 0.11, 0.18, 0.14, 0.12]

Sample Log Entries:
  patient_id risk_level  latency_sec  tokens_used  anomaly
0     H10000        LOW         0.45           90    False
1     H10001     MEDIUM         0.37           90    False
2     H10002        LOW         0.49          109    False
3     H10003        LOW         0.34           97    False
4     H10004     MEDIUM         0.32           87    False
5     H10005        LOW  

In [ ]:
# # Step 9: LLMOps Logging & Drift Monitoring - simulated

# import random


# random.seed(42)
# log_records = []

# for _, row in summaries_df.iterrows():
#     latency = round(random.uniform(0.8, 2.1), 2)
#     confidence = round(random.uniform(0.75, 0.97), 2)
#     tokens = random.randint(180, 280)

#     log_records.append({
#         'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
#         'patient_id': row['patient_id'],
#         'risk_score': row['risk_score'],
#         'risk_level': row['risk_level'],
#         'anomaly': row['anomaly'],
#         'silent_risk': row['silent_risk'],
#         'latency_sec': latency,
#         'confidence': confidence,
#         'tokens_used': tokens,
#         'model': 'llama-3.1-8b-instant'
#     })

# log_df = pd.DataFrame(log_records)

# # Drift monitoring
# drift_scores = [0.08, 0.09, 0.13, 0.11, 0.18, 0.14, 0.12]
# drift_status = 'STABLE' if max(drift_scores) < 0.25 else 'ALERT'

# print("=" * 50)
# print("LLMOPS PIPELINE SUMMARY")
# print("=" * 50)
# print(f"Total inferences run     : {len(log_df)}")
# print(f"Model                    : llama-3.1-8b-instant")
# print(f"Embedding model          : all-MiniLM-L6-v2 (384 dimensions)")
# print(f"Vector store             : FAISS (IndexFlatL2)")
# print(f"Avg latency              : {log_df['latency_sec'].mean():.2f}s")
# print(f"Avg confidence           : {log_df['confidence'].mean()*100:.0f}%")
# print(f"Avg tokens per call      : {log_df['tokens_used'].mean():.0f}")
# print(f"Anomalies flagged        : {log_df['anomaly'].sum()}")
# print(f"Silent risk profiles     : {log_df['silent_risk'].sum()}")
# print(f"Embedding drift (latest) : {drift_scores[-1]}")
# print(f"Drift status             : {drift_status} (threshold 0.25)")
# print(f"Drift scores across runs : {drift_scores}")
# print(f"\nSample Log Entries:")
# print(log_df[['patient_id', 'risk_level', 'latency_sec', 
#               'confidence', 'tokens_used', 'anomaly']].head(10))

In [ ]:
# # Step 9: LLMOps Logging & Drift Monitoring

# random.seed(42)
# log_records = []

# for _, row in summaries_df.iterrows():
#     latency = round(random.uniform(0.8, 2.1), 2)
#     confidence = round(random.uniform(0.75, 0.97), 2)
#     tokens = random.randint(180, 280)

#     log_records.append({
#         'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
#         'patient_id': row['patient_id'],
#         'risk_score': row['risk_score'],
#         'risk_level': row['risk_level'],
#         'anomaly': row['anomaly'],
#         'silent_risk': row['silent_risk'],
#         'latency_sec': latency,
#         'confidence': confidence,
#         'tokens_used': tokens,
#         'model': 'llama-3.1-8b-instant'
#     })

# log_df = pd.DataFrame(log_records)

# # Drift monitoring
# drift_scores = [0.08, 0.09, 0.13, 0.11, 0.18, 0.14, 0.12]
# drift_status = 'STABLE' if max(drift_scores) < 0.25 else 'ALERT'

# print("=" * 50)
# print("LLMOPS PIPELINE SUMMARY")
# print("=" * 50)
# print(f"Total inferences run     : {len(log_df)}")
# print(f"Model                    : llama-3.1-8b-instant")
# print(f"Embedding model          : all-MiniLM-L6-v2 (384 dimensions)")
# print(f"Vector store             : FAISS (IndexFlatL2)")
# print(f"Avg latency              : {log_df['latency_sec'].mean():.2f}s")
# print(f"Avg confidence           : {log_df['confidence'].mean()*100:.0f}%")
# print(f"Avg tokens per call      : {log_df['tokens_used'].mean():.0f}")
# print(f"Anomalies flagged        : {log_df['anomaly'].sum()}")
# print(f"Silent risk profiles     : {log_df['silent_risk'].sum()}")
# print(f"Embedding drift (latest) : {drift_scores[-1]}")
# print(f"Drift status             : {drift_status} (threshold 0.25)")
# print(f"Drift scores across runs : {drift_scores}")
# print(f"\nSample Log Entries:")
# print(log_df[['patient_id', 'risk_level', 'latency_sec', 
#               'confidence', 'tokens_used', 'anomaly']].head(10))

## Step 10: Final Output & Summary Report

In [18]:
# Step 10: Final Output & Summary Report

final_output = df[['patient_id', 'age', 'sex', 'bmi', 'smoker',
                    'diabetic', 'dx_code', 'risk_score',
                    'risk_level', 'anomaly', 'silent_risk']].copy()

# Add summaries
final_output = final_output.merge(
    summaries_df[['patient_id', 'summary']],
    on='patient_id',
    how='left'
)

# Strip any LLM meta-commentary from summaries
final_output['summary'] = final_output['summary'].str.replace(
    r'\(Note:.*?\)', '', regex=True
).str.replace(
    r'\*Note:.*', '', regex=True
).str.strip()

final_output['summary'] = final_output['summary'].fillna('No summary generated.')

# Save to CSV
final_output.to_csv('health_risk_predictions_v2.csv', index=False)

total = len(final_output)
high = (final_output['risk_level'] == 'HIGH').sum()
medium = (final_output['risk_level'] == 'MEDIUM').sum()
low = (final_output['risk_level'] == 'LOW').sum()

print("=" * 50)
print("CAPSTONE PROJECT — HEALTH RISK PREDICTION SYSTEM")
print("=" * 50)
print(f"Vertical                 : Healthcare")
print(f"Dataset                  : healthcare_patients.csv + healthcare_encounters.csv")
print(f"Embedding Model          : all-MiniLM-L6-v2 (Sentence Transformers)")
print(f"Vector Store             : FAISS IndexFlatL2 — {index.ntotal} vectors, {embeddings.shape[1]} dimensions")
print(f"LLM Model                : llama-3.1-8b-instant via Groq API")
print(f"RAG                      : FAISS similarity search — top 3 similar patients per inference")
print(f"Anomaly Detection        : Z-score threshold 2.5σ on BMI and risk score")
print(f"Prompt Engineering       : Business-audience focused, 3-sentence plain language output")
print(f"LLMOps                   : Real latency and token tracking + KL-divergence drift monitoring (illustrative)")
print(f"Confidence Scoring       : Not available — logprobs not supported by llama-3.1-8b-instant on Groq")
print()
print("RESULTS SUMMARY")
print("-" * 50)
print(f"Total patients processed : {total}")
print(f"HIGH risk patients       : {high}  ({high/total*100:.2f}%)")
print(f"MEDIUM risk patients     : {medium} ({medium/total*100:.2f}%)")
print(f"LOW risk patients        : {low} ({low/total*100:.2f}%)")
print(f"Anomalies detected       : {final_output['anomaly'].sum()}")
print(f"Silent risk profiles     : {final_output['silent_risk'].sum()}")
print(f"LLM summaries generated  : {len(summaries_df)}")
print(f"Avg real latency         : {log_df['latency_sec'].mean():.2f}s")
print(f"Avg real tokens per call : {log_df['tokens_used'].mean():.0f}")
print(f"Embedding drift status   : {drift_status} ({drift_scores[-1]} / threshold 0.25)")
print()
print("OUTPUT FILES")
print("-" * 50)
print("health_risk_predictions_v2.csv — full scored dataset with summaries")
print()
print("Notebook ready for HTML export.")
print("File > Export > HTML to generate submission file.")

CAPSTONE PROJECT — HEALTH RISK PREDICTION SYSTEM
Vertical                 : Healthcare
Dataset                  : healthcare_patients.csv + healthcare_encounters.csv
Embedding Model          : all-MiniLM-L6-v2 (Sentence Transformers)
Vector Store             : FAISS IndexFlatL2 — 400 vectors, 384 dimensions
LLM Model                : llama-3.1-8b-instant via Groq API
RAG                      : FAISS similarity search — top 3 similar patients per inference
Anomaly Detection        : Z-score threshold 2.5σ on BMI and risk score
Prompt Engineering       : Business-audience focused, 3-sentence plain language output
LLMOps                   : Real latency and token tracking + KL-divergence drift monitoring (illustrative)
Confidence Scoring       : Not available — logprobs not supported by llama-3.1-8b-instant on Groq

RESULTS SUMMARY
--------------------------------------------------
Total patients processed : 400
HIGH risk patients       : 27  (6.75%)
MEDIUM risk patients     : 195 (48.75%

In [ ]:
# # Step 10: Final Output & Summary Report - simulated

# final_output = df[['patient_id', 'age', 'sex', 'bmi', 'smoker',
#                     'diabetic', 'dx_code', 'risk_score',
#                     'risk_level', 'anomaly', 'silent_risk']].copy()

# # Add summaries
# final_output = final_output.merge(
#     summaries_df[['patient_id', 'summary']],
#     on='patient_id',
#     how='left'
# )
# final_output['summary'] = final_output['summary'].fillna('Low risk — no summary generated.')

# # Save to CSV
# final_output.to_csv('health_risk_predictions_v2.csv', index=False)

# # Summary
# total = len(final_output)
# high = (final_output['risk_level'] == 'HIGH').sum()
# medium = (final_output['risk_level'] == 'MEDIUM').sum()
# low = (final_output['risk_level'] == 'LOW').sum()

# print("=" * 50)
# print("CAPSTONE PROJECT — HEALTH RISK PREDICTION SYSTEM")
# print("=" * 50)
# print(f"Vertical                 : Healthcare")
# print(f"Dataset                  : healthcare_patients.csv + healthcare_encounters.csv")
# print(f"Embedding Model          : all-MiniLM-L6-v2 (Sentence Transformers)")
# print(f"Vector Store             : FAISS IndexFlatL2 — {index.ntotal} vectors, {embeddings.shape[1]} dimensions")
# print(f"LLM Model                : llama-3.1-8b-instant via Groq API")
# print(f"RAG                      : FAISS similarity search — top 3 similar patients per inference")
# print(f"Anomaly Detection        : Z-score threshold 2.5σ on BMI and risk score")
# print(f"Prompt Engineering       : Business-audience focused, 3-sentence plain language output")
# print(f"LLMOps                   : Latency, confidence, token tracking + KL-divergence drift monitoring")
# print()
# print("RESULTS SUMMARY")
# print("-" * 50)
# print(f"Total patients processed : {total}")
# print(f"HIGH risk patients       : {high}  ({high/total*100:.2f}%)")
# print(f"MEDIUM risk patients     : {medium} ({medium/total*100:.2f}%)")
# print(f"LOW risk patients        : {low} ({low/total*100:.2f}%)")
# print(f"Anomalies detected       : {final_output['anomaly'].sum()}")
# print(f"Silent risk profiles     : {final_output['silent_risk'].sum()}")
# print(f"LLM summaries generated  : {len(summaries_df)}")
# print(f"Avg model confidence     : {log_df['confidence'].mean()*100:.0f}%")
# print(f"Embedding drift status   : {drift_status} ({drift_scores[-1]} / threshold 0.25)")
# print()
# print("OUTPUT FILES")
# print("-" * 50)
# print("health_risk_predictions_v2.csv — full scored dataset with summaries")
# print()
# print("Notebook ready for HTML export.")
# print("File > Export > HTML to generate submission file.")

In [19]:
# See output sample
sample_output = pd.read_csv("health_risk_predictions_v2.csv")

print("Shape:", sample_output.shape)
print("\nFirst 10 rows:")
print(sample_output.head(10).to_string(index=False))

print("\nSelected columns sample:")
print(sample_output[['patient_id', 'risk_score', 'risk_level', 'anomaly', 'silent_risk', 'summary']].head(5).to_string(index=False))


Shape: (400, 12)

First 10 rows:
patient_id  age sex  bmi  smoker  diabetic dx_code  risk_score risk_level  anomaly  silent_risk                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   summary
    H10000   68   M 25.3       0         0     E78        0.35        LOW    False        False                                                                            Based on our analysis, Patient H10000 has a low risk score of 0.35, indicating a relatively low likelihood of a significant health issue. However,

In [20]:
# Step 10b: Before vs After sample audit (input -> risk/anomaly/silent-risk/summary)

audit_df = df.merge(
    summaries_df[['patient_id', 'summary']],
    on='patient_id',
    how='left'
).copy()

audit_df['summary'] = audit_df['summary'].fillna('Low risk — no summary generated.')

# Build a combined "before" input view
audit_df['combined_input'] = audit_df.apply(
    lambda r: (
        f"Patient {r['patient_id']}, age {r['age']}, sex {r['sex']}, "
        f"BMI {r['bmi']}, smoker {r['smoker']}, diabetic {r['diabetic']}, "
        f"dx_code {r['dx_code']}. Note: {r['note']}"
    ),
    axis=1
)

# Select representative samples:
# - risk levels (HIGH/MEDIUM/LOW)
# - anomaly cases
# - silent risk cases
samples = pd.concat([
    audit_df[audit_df['risk_level'] == 'HIGH'].head(2),
    audit_df[audit_df['risk_level'] == 'MEDIUM'].head(2),
    audit_df[audit_df['risk_level'] == 'LOW'].head(2),
    audit_df[audit_df['anomaly'] == True].head(3),
    audit_df[audit_df['silent_risk'] == True].head(3)
], ignore_index=True).drop_duplicates(subset=['patient_id'])

# Final before/after view
before_after = samples[[
    'patient_id',
    'combined_input',      # BEFORE
    'risk_score',
    'risk_level',
    'anomaly',
    'silent_risk',
    'summary'              # AFTER
]].copy()

print(f"Selected sample rows: {len(before_after)}")
print(before_after.to_string(index=False, max_colwidth=140))

# Optional: save for report
before_after.to_csv('before_after_sample_v2.csv', index=False)
print("\nSaved: before_after_sample_v2.csv")
# ...existing code...

Selected sample rows: 8
patient_id                                                                                                                               combined_input  risk_score risk_level  anomaly  silent_risk                                                                                                                                      summary
    H10009 Patient H10009, age 67, sex M, BMI 31.4, smoker 1, diabetic 1, dx_code E11. Note: Follow-up visit. asthma controlled. Recommending lifest...        0.90       HIGH     True        False Patient H10009 is at high risk for potential health complications and has a high chance of increasing healthcare costs due to his age, BM...
    H10012 Patient H10012, age 72, sex M, BMI 28.4, smoker 1, diabetic 1, dx_code J45. Note: Patient presents with indigestion for 5 days. History o...        0.85       HIGH     True         True Patient H10012 presents a high risk of potential complications due to existing health conditions, inclu

In [21]:
# Step 10b: Before vs After sample audit (input -> risk/anomaly/silent-risk/summary) - detailed with no truncation

from IPython.display import display

# Keep full text visible
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', None)

print(f"Selected sample rows: {len(before_after)}")

# Better notebook display (no truncation + left aligned text)
display(
    before_after.style
    .set_properties(subset=['combined_input', 'summary'], **{
        'text-align': 'left',
        'white-space': 'pre-wrap'
    })
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
)

# Optional: full untruncated plain-text view (record-by-record)
for i, r in before_after.iterrows():
    print("\n" + "=" * 120)
    print(f"Patient ID  : {r['patient_id']}")
    print(f"Risk Score  : {r['risk_score']} ({r['risk_level']})")
    print(f"Anomaly     : {r['anomaly']}")
    print(f"Silent Risk : {r['silent_risk']}")
    print(f"BEFORE      : {r['combined_input']}")
    print(f"AFTER       : {r['summary']}")

# Optional: save for report
before_after.to_csv('before_after_sample_v2.csv', index=False)
print("\nSaved: before_after_sample_v2.csv")



Selected sample rows: 8


,patient_id,combined_input,risk_score,risk_level,anomaly,silent_risk,summary
0,H10009,"Patient H10009, age 67, sex M, BMI 31.4, smoker 1, diabetic 1, dx_code E11. Note: Follow-up visit. asthma controlled. Recommending lifestyle changes.",0.900000,HIGH,True,False,"Patient H10009 is at high risk for potential health complications and has a high chance of increasing healthcare costs due to his age, BMI, smoking status, and diabetes. Historically, similar patients with similar characteristics have seen an increase in health issues, suggesting that this patient could experience similar health challenges unless proactive steps are taken. Our analysis recommends implementing preventive measures and close monitoring to mitigate this risk and ensure optimal patient outcomes. (Note: This plain language summary highlights the key points the business team needs to act on, focusing on potential financial impact and the need for proactive measures to manage the risk.)"
1,H10012,"Patient H10012, age 72, sex M, BMI 28.4, smoker 1, diabetic 1, dx_code J45. Note: Patient presents with indigestion for 5 days. History of diabetes. Vitals stable.",0.850000,HIGH,True,True,"Patient H10012 presents a high risk of potential complications due to existing health conditions, including diabetes. While their symptoms are currently mild, their risk profile suggests that we should be prepared for possible escalation. A closer examination of the patient's medical history, current treatment plan, and potential interventions will be necessary to mitigate this risk."
2,H10001,"Patient H10001, age 70, sex M, BMI 23.4, smoker 1, diabetic 0, dx_code J45. Note: Reports headache episodically; denies chest pain. On meds for sciatica.",0.600000,MEDIUM,False,True,"Based on the analysis, Patient H10001 has a medium risk profile (0.6) and has not been flagged as an anomaly. However, our team has identified some historical cases with similar profiles that present potential concerns, such as episodes of headache, back pain, or anxiety. As a result, we recommend that the underwriting and clinical teams closely review this patient's case and monitor their condition to ensure timely interventions and prevent potential complications."
3,H10004,"Patient H10004, age 77, sex F, BMI 33.3, smoker 0, diabetic 0, dx_code F41. Note: Patient presents with insomnia for 12 days. History of diabetes. Vitals stable.",0.500000,MEDIUM,False,False,"Based on our analysis, patient H10004 has a medium risk score of 0.5, indicating a moderate level of concern for potential future health complications. This risk is elevated due to her insomnia diagnosis, which may require further monitoring and management to prevent escalation, despite having no history of diabetes. Our analysis suggests a cautious approach to this patient, recommending ongoing assessment and potential adjustments to her treatment plan to minimize future health risks."
4,H10000,"Patient H10000, age 68, sex M, BMI 25.3, smoker 0, diabetic 0, dx_code E78. Note: Patient presents with headache for 1 days. History of diabetes. Vitals stable.",0.350000,LOW,False,False,"Based on our analysis, Patient H10000 has a low risk score of 0.35, indicating a relatively low likelihood of a significant health issue. However, we've identified a few similar cases with slightly higher risk scores that may warrant closer review of this patient's medical history and treatment plan, particularly given their history of diabetes. We recommend closely monitoring Patient H10000's condition and adjusting their care plan if necessary to mitigate potential risks."
5,H10002,"Patient H10002, age 41, sex M, BMI 24.5, smoker 0, diabetic 0, dx_code M54. Note: Follow-up visit. hyperlipidemia controlled. Recommending lifestyle changes.",0.200000,LOW,False,False,"Here's a 3-sentence plain language risk summary for the business team: Patient H10002 has a low risk score of 0.2, indicating that this patient is unlikely to require significant medical intervention. However, 


Patient ID  : H10009
Risk Score  : 0.9 (HIGH)
Anomaly     : True
Silent Risk : False
BEFORE      : Patient H10009, age 67, sex M, BMI 31.4, smoker 1, diabetic 1, dx_code E11. Note: Follow-up visit. asthma controlled. Recommending lifestyle changes.
AFTER       : Patient H10009 is at high risk for potential health complications and has a high chance of increasing healthcare costs due to his age, BMI, smoking status, and diabetes. Historically, similar patients with similar characteristics have seen an increase in health issues, suggesting that this patient could experience similar health challenges unless proactive steps are taken. Our analysis recommends implementing preventive measures and close monitoring to mitigate this risk and ensure optimal patient outcomes.

(Note: This plain language summary highlights the key points the business team needs to act on, focusing on potential financial impact and the need for proactive measures to manage the risk.)

Patient ID  : H10012
Risk Sco